**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**09 | Full-Development Grouped Cross-Validation**

![Stage 09 scientific workflow](../../images/09_workflow.svg)

**Objective:** Compare three classical classifiers across five participant-disjoint Development folds and examine fold-level and paired participant-level performance.

**Dataset:** All 29 HARTH v2.0 Development participants, using the time-safe, 30-feature window cohort defined in Notebook 08.

**Notebook Scope:** Train and evaluate models within Development only; compare Random Forest and Logistic Regression on the same held-out participants. S023 and S032 remain excluded from this Notebook. Earlier exploratory inspection of those reserved recordings means this is not evidence of strict blind final Test evaluation.

**01. Imports and Reusable Project Functions**

In [ ]:
# ==================== 01. Imports and Project Environment ====================

# ------------ 1. Import Python tools for portable project paths. -------------
import sys
from pathlib import Path

# ------------ 2. Import NumPy and pandas for cohort data tables. -------------
import numpy as np
import pandas as pd

# ----------- 3. Import Matplotlib and notebook display functions. ------------
import matplotlib.pyplot as plt
from IPython.display import display

# ---------- 4. Locate repository root without fixed machine paths. -----------
root = Path.cwd().resolve()
while not (root / "python_modules/core/data_loading.py").is_file():
    if root.parent == root:
        raise RuntimeError("Open this notebook inside human-movement-intelligence.")
    root = root.parent

# ---------- 5. Register shared Python and HARTH study directories. -----------
sys.path.insert(0, str(root / "python_modules"))
sys.path.insert(0, str(root / "wearable_activity_recognition/code/src"))

# ---------- 6. Import original HARTH data and GroupKFold utilities. ----------
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, prepare_harth_feature_cohort
from core.data_splitting import audit_group_class_coverage

# ---------- 7. Import CV evaluation and paired subject comparison. -----------
from core.group_cross_validation import (
    evaluate_group_cross_validation, compare_models_by_subject,
)

# -------------- 8. Apply the approved academic plotting style. ---------------
from core.plot_style import plot_style
plot_style()

# ---------- 9. Confirm the original research components are ready. -----------
print("HARTH full-development CV components: ready")

**02. Reconstruct the Complete 29-Person Development Cohort**

Reuse the source cohort and windowing settings from Notebook 08. Filter the eight-class task after constructing time-safe windows; retrieve cached recordings or download missing files as needed.

In [ ]:
# ============== 02. Reconstruct Full Development Feature Cohort ==============

# ------------- 1. Fix the two original reserved Test identities. -------------
RESERVED_TEST = {"S023", "S032"}

# ---------- 2. Identify the remaining 29 Development participants. -----------
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)

# ------------ 3. Define the documented eight-class activity task. ------------
TASK_CLASSES = tuple(range(1, 9))

# ---------- 4. Check cohort size and absence of reserved Test IDs. -----------
assert len(DEVELOPMENT_SUBJECTS) == 29
assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)

# --------- 5. Recreate label-pure windows and statistical features. ----------
features, metadata, overview, excluded = prepare_harth_feature_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    window_size=100, step_size=50, sampling_rate=50,
)

# --------- 6. Verify one feature vector per source metadata record. ----------
assert len(features) == len(metadata)

# ---------- 7. Report the observed 29-person feature-matrix shape. -----------
print("Development feature matrix:", features.shape)

# ----------- 8. Report documented exclusions and Test non-access. ------------
print("Excluded out-of-task windows:", int(overview["excluded_windows"].sum()))
print("Reserved Test participants loaded: NO")

**03. Verify Subject-Disjoint Folds and Class Coverage**

Check that Train and Validation have no participant overlap and that the predefined classes are represented in each fold. The saved fold audit reports complete class coverage, but this alone does not establish model performance.

In [ ]:
# =============== 03. Audit All Five Participant-Disjoint Folds ===============

# ------- 1. Inspect class coverage and subject-group fold feasibility. -------
coverage, fold_audit = audit_group_class_coverage(
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES, n_splits=5,
)

# ----------- 2. Display fold membership and source-window counts. ------------
display(fold_audit[[
    "fold", "train_subjects", "validation_subjects",
    "train_windows", "validation_windows",
    "missing_train_classes", "missing_validation_classes",
]])

# ---------- 3. Require all eight task classes in every Train fold. -----------
assert not fold_audit["missing_train_classes"].map(bool).any()

# -------- 4. Require all eight task classes in every Validation fold. --------
assert not fold_audit["missing_validation_classes"].map(bool).any()

# --------------- 5. Confirm the source-data fold audit passed. ---------------
print("Fold class-coverage audit: PASSED")

**04. Compare Three Models Across Five Development Folds**

Refit each classifier within every fold. Logistic Regression scaling uses that fold's Train observations only. Reserved Test recordings are not loaded.

In [ ]:
# =============== 04. Evaluate Classical Models with GroupKFold ===============

# ------- 1. Fit three classical models across five Development folds. --------
fold_results, subject_results, summary = evaluate_group_cross_validation(
    features=features.to_numpy(),
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    n_splits=5, random_state=42,
)

# ----------- 2. Display saved cross-fold model summary statistics. -----------
display(summary.round(4))

# ---------- 3. Inspect original held-out subject and fold metrics. -----------
display(fold_results[[
    "fold", "model", "validation_subjects",
    "validation_windows", "validation_classes_present",
    "macro_f1", "balanced_accuracy",
]].round(4))

**05. Visualise Variation in Held-Out Macro F1**

Compare fold-level Macro F1 across models, accounting for differences in participant composition and window counts. These folds share the Development cohort and do not represent independent replications.

In [ ]:
# ================= 05. Visualise Five-Fold Model Performance =================

# ---------- 1. Initialise the original fold-level comparison plot. -----------
fig, ax = plt.subplots(figsize=(10, 5))

# ---------- 2. Plot Macro F1 by held-out fold for each classifier. -----------
for model_name, table in fold_results.groupby("model", sort=False):
    table = table.sort_values("fold")
    ax.plot(table["fold"], table["macro_f1"], marker="o",
            label=f"{model_name} (mean={table['macro_f1'].mean():.3f})")

ax.set(title="HARTH: 29-person development grouped CV",
       xlabel="Held-out fold", ylabel="Macro F1",
       ylim=(0, 1), xticks=[1, 2, 3, 4, 5])
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

**06. Compare Random Forest and Logistic Regression by Person**

Pair fixed eight-class Macro F1 scores for the same 29 held-out Development participants. The metric `difference_a_minus_b` is Random Forest minus Logistic Regression; positive values favour Random Forest.

In [ ]:
# ============== 06. Pair Random Forest and Logistic Regression ===============

# -------- 1. Pair two model scores for identical Development people. ---------
paired, paired_summary = compare_models_by_subject(
    subject_results,
    model_a="Random Forest",
    model_b="Logistic Regression",
    score_column="macro_f1_fixed_task",
)

# --------- 2. Display signed subject-wise fixed-task Macro F1 gaps. ----------
display(paired.round(4))

# ------------ 3. Display the original paired participant summary. ------------
display(pd.DataFrame([paired_summary]))

# ------------ 4. Initialise the original signed-difference chart. ------------
fig, ax = plt.subplots(figsize=(13, 5))

# --------- 5. Encode positive and negative differences consistently. ---------
bar_colours = ["tab:blue" if value >= 0 else "tab:orange"
               for value in paired["difference_a_minus_b"]]

# ---------- 6. Plot paired Random Forest minus Logistic Regression. ----------
ax.bar(paired["subject_id"], paired["difference_a_minus_b"], color=bar_colours)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Per-subject Macro F1 difference: Random Forest minus Logistic Regression",
       xlabel="Held-out development participant", ylabel="Paired Macro F1 difference")
ax.tick_params(axis="x", rotation=70)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

**07. Compare Activity Support and Individual Model Scores**

Report class availability and fixed-task Macro F1 for each participant. Interpret individual differences alongside activity support rather than relying only on aggregate scores.

In [ ]:
# =============== 07. Compare Source Class Support and Results ================

# -------- 1. Count eligible windows and observed classes per person. ---------
person_support = pd.DataFrame({
    "subject_id": coverage.index,
    "eligible_windows": coverage.sum(axis=1).to_numpy(),
    "observed_classes": (coverage > 0).sum(axis=1).to_numpy(),
})

# ------------ 2. Align fixed-eight-class Macro F1 across models. -------------
comparison = subject_results.pivot(
    index="subject_id", columns="model", values="macro_f1_fixed_task"
).reset_index()

# ---------- 3. Join source coverage and model scores by subject ID. ----------
participant_table = person_support.merge(comparison, on="subject_id", how="left")

# --------- 4. Display the original participant-level evidence table. ---------
display(participant_table.round(4))

# --------- 5. Verify all 29 Development individuals are represented. ---------
print("Independent development subjects represented:", len(participant_table))

**08. Scientific Quality Gate and Evidence Boundaries**

Verify the five-fold results, participant separation, source integrity and reserved Test exclusion. Because some properties of S023 and S032 were inspected in earlier exploratory work, do not describe the broader research as strictly preregistered and blind.

In [ ]:
# ============ 08. Development GroupKFold Scientific Quality Gate =============

# ------------- 1. Check all Development subject IDs are unique. --------------
assert len(DEVELOPMENT_SUBJECTS) == len(set(DEVELOPMENT_SUBJECTS)) == 29

# ------------- 2. Exclude historically reserved Test identities. -------------
assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)

# ----------- 3. Confirm results cover all 29 Development subjects. -----------
assert len(subject_results["subject_id"].unique()) == 29

# --------- 4. Ensure original Test people have no evaluated scores. ----------
assert not (set(subject_results["subject_id"]) & RESERVED_TEST)

# --------- 5. Require results for five folds and three classifiers. ----------
assert len(fold_results) == 5 * 3

# ----------- 6. Verify the paired model comparison uses 29 people. -----------
assert len(paired) == 29

# ------------ 7. Check unique person-by-model evaluation records. ------------
assert subject_results.groupby(["model", "subject_id"]).size().eq(1).all()

# ------ 8. Confirm all source-derived statistical features are finite. -------
assert np.isfinite(features.to_numpy()).all()

# ------------- 9. Report passing Development CV quality checks. --------------
print("Full-development GroupKFold: PASSED")

# ------------ 10. State that no reserved Test data were accessed. ------------
print("Original final Test loaded or scored: NO")

# ----------- 11. Maintain the original limits on benchmark claims. -----------
print("Final independent benchmark claim: NOT SUPPORTED")